Retrieval Augmented Generation (RAG) Bot

In [2]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

"""""
img = mpimg.imread('process.png')
plt.imshow(img)
plt.axis('off')
"""

'""\nimg = mpimg.imread(\'process.png\')\nplt.imshow(img)\nplt.axis(\'off\')\n'

Content Extraction 

In [10]:

import pdfplumber
import os

PDF_FOLDER = os.path.join(os.getcwd(), 'raw_documents')
OUTPUT_FOLDER = os.path.join(os.getcwd(), "extracted_texts")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def extract_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + "\n"
    return text

for filename in os.listdir(PDF_FOLDER):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(PDF_FOLDER, filename)
        print("Extracting:", pdf_path)

        text = extract_pdf(pdf_path)

        # Save text file for this PDF
        output_path = os.path.join(OUTPUT_FOLDER, filename.replace(".pdf", ".txt"))
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)



Extracting: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\raw_documents\Topic 1_ Pre-16th Century Philippines Reading Materials.pdf


Chunking

In [11]:
import os
import json
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import TokenTextSplitter

# ============================================================
#                   HYPERPARAMETER CONFIG
# ============================================================
CONFIG = {
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",

    # Semantic Chunking Parameters
    "semantic_min_chunk_size":        300,
    "semantic_breakpoint_type":       "percentile",
    "semantic_breakpoint_amount":     90,

    # Token-based Chunking Parameters
    "token_chunk_size":   600,
    "token_overlap":       75,
    "tokenizer_name":     "cl100k_base",

    # Paths
    "text_folder":   os.path.join(os.getcwd(), "extracted_texts"),
    "chunks_folder": os.path.join(os.getcwd(), "chunks_json"),
}
os.makedirs(CONFIG["chunks_folder"], exist_ok=True)
# ============================================================


# -------------------- LOAD EMBEDDINGS --------------------
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["model_name"]
)

# -------------------- SEMANTIC CHUNKER --------------------
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type=CONFIG["semantic_breakpoint_type"],
    breakpoint_threshold_amount=CONFIG["semantic_breakpoint_amount"],
    min_chunk_size=CONFIG["semantic_min_chunk_size"],
)

# -------------------- TOKEN SPLITTER --------------------
token_splitter = TokenTextSplitter(
    chunk_size=CONFIG["token_chunk_size"],
    chunk_overlap=CONFIG["token_overlap"],
    encoding_name=CONFIG["tokenizer_name"]
)

# ============================================================
#                   PROCESSING LOOP
# ============================================================
for filename in os.listdir(CONFIG["text_folder"]):
    if filename.endswith(".txt"):
        file_path = os.path.join(CONFIG["text_folder"], filename)
        print("Chunking:", file_path)

        # Load extracted text
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # Stage 1: Semantic chunking
        semantic_chunks = semantic_chunker.split_text(text)

        # Stage 2: Token chunking
        final_chunks = []
        for sc in semantic_chunks:
            final_chunks.extend(token_splitter.split_text(sc))

        # Save JSON output
        base_name = filename.replace(".txt", "")
        json_path = os.path.join(CONFIG["chunks_folder"], f"{base_name}_chunks.json")

        json_output = [
            {
                "doc_id": base_name,
                "chunk_id": f"{base_name}_chunk_{i}",
                "chunk_index": i,
                "text": chunk
            }
            for i, chunk in enumerate(final_chunks)
        ]

        with open(json_path, "w", encoding="utf-8") as out:
            json.dump(json_output, out, indent=4, ensure_ascii=False)

        print(f"Saved {len(final_chunks)} chunks → {json_path}")


ImportError: cannot import name 'HuggingFaceEmbeddings' from 'langchain.embeddings' (c:\Users\Migs\Desktop\ragbot\myenv\Lib\site-packages\langchain\embeddings\__init__.py)

In [1]:
import os
import json
# NEW: Specialized import paths
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import TokenTextSplitter

# ============================================================
#                   HYPERPARAMETER CONFIG
# ============================================================
CONFIG = {
    # Using the updated HuggingFace model pathing
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",

    # Semantic Chunking Parameters
    "semantic_min_chunk_size":        300,
    "semantic_breakpoint_type":       "percentile",
    "semantic_breakpoint_amount":     95,

    # Token-based Chunking Parameters
    "token_chunk_size":   600,
    "token_overlap":       75,
    "tokenizer_name":     "cl100k_base",

    # Paths
    "text_folder":   os.path.join(os.getcwd(), "extracted_texts"),
    "chunks_folder": os.path.join(os.getcwd(), "chunks_json"),
}
os.makedirs(CONFIG["chunks_folder"], exist_ok=True)

# -------------------- LOAD EMBEDDINGS --------------------
# Updated to use langchain_huggingface class
embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["model_name"]
)

# -------------------- SEMANTIC CHUNKER --------------------
semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type=CONFIG["semantic_breakpoint_type"],
    breakpoint_threshold_amount=CONFIG["semantic_breakpoint_amount"],
    # min_chunk_size is supported in recent experimental versions
)

# -------------------- TOKEN SPLITTER --------------------
# Updated to use langchain_text_splitters class
token_splitter = TokenTextSplitter(
    chunk_size=CONFIG["token_chunk_size"],
    chunk_overlap=CONFIG["token_overlap"],
    encoding_name=CONFIG["tokenizer_name"]
)

# ============================================================
#                   PROCESSING LOOP
# ============================================================
for filename in os.listdir(CONFIG["text_folder"]):
    if filename.endswith(".txt"):
        file_path = os.path.join(CONFIG["text_folder"], filename)
        print(f"Chunking: {file_path}")

        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # Stage 1: Semantic chunking (Now returns List[str] or List[Document])
        semantic_chunks = semantic_chunker.split_text(text)

        # Stage 2: Token chunking for safety/uniformity
        final_chunks = []
        for sc in semantic_chunks:
            final_chunks.extend(token_splitter.split_text(sc))

        # Save JSON output
        base_name = filename.replace(".txt", "")
        json_path = os.path.join(CONFIG["chunks_folder"], f"{base_name}_chunks.json")

        json_output = [
            {
                "doc_id": base_name,
                "chunk_id": f"{base_name}_chunk_{i}",
                "chunk_index": i,
                "text": chunk
            }
            for i, chunk in enumerate(final_chunks)
        ]

        with open(json_path, "w", encoding="utf-8") as out:
            json.dump(json_output, out, indent=4, ensure_ascii=False)

        print(f"Saved {len(final_chunks)} chunks → {json_path}")

c:\Users\Migs\Desktop\ragbot\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunking: c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\extracted_texts\Topic 1_ Pre-16th Century Philippines Reading Materials.txt
Saved 35 chunks → c:\Users\Migs\Desktop\ragbot\backend\app\services\evaluation\chunks_json\Topic 1_ Pre-16th Century Philippines Reading Materials_chunks.json


Local Eval Dataset Creation

In [ ]:
from langchain.llms import Ollama

llm = Ollama(model="mistral:instruct")

import json
import os
import re

CHUNKS_FOLDER = "chunks_json"
OUTPUT_FILE = "mistral_rag_eval.json"

def extract_json(text):
    try:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print("❌ JSON parsing error:", e)
    return []

all_samples = []

for file_name in os.listdir(CHUNKS_FOLDER):
    if file_name.endswith(".json"):
        path = os.path.join(CHUNKS_FOLDER, file_name)
        with open(path, "r", encoding="utf-8") as f:
            chunks = json.load(f)
        
        for chunk in chunks:
            chunk_text = f"Chunk ID: {chunk['chunk_id']}\nText:\n{chunk['text']}\n"
            prompt = f"""
You are building a RAG evaluation dataset.

Here is one text chunk:

{chunk_text}

Task:
1. Generate 1 question answerable from this chunk.
2. Provide the answer.
3. Specify the chunk ID(s) necessary to answer.

Return output as JSON:
[{{"query": "...", "answer": "...", "relevant_chunks": ["{chunk['chunk_id']}"]}}]
"""
            try:
                response = llm(prompt)
                data = extract_json(response)
                if data:
                    all_samples.extend(data)
            except Exception as e:
                print(f"❌ Error for chunk {chunk['chunk_id']}: {e}")

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    json.dump(all_samples, out, indent=4, ensure_ascii=False)

print(f"\n✅ Mistral RAG evaluation dataset saved to {OUTPUT_FILE}")
print(f"Total samples: {len(all_samples)}")




Hosted Inference Eval Dataset Creation

In [5]:
import json
import os
from together import Together

import random

# ----------------------------
# CONFIG
# ----------------------------
TOGETHER_API_KEY = "f093074f102974466d625db36d8bd171b92df916fa78eb7b91faa9108e6ed5c2"
CHUNKS_FOLDER = "chunks_json"
OUTPUT_FILE = "rag_eval.json"
MODEL_NAME =  "meta-llama/Llama-3.3-70B-Instruct-Turbo"
MAX_TOKENS = 512

# ----------------------------
# INITIALIZE CLIENT
# ----------------------------
client = Together(api_key=TOGETHER_API_KEY)

# ----------------------------
# HELPER FUNCTION
# ----------------------------
def extract_json(text):
    import re, json
    try:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception as e:
        print("❌ JSON parsing error:", e)
    return []

# ----------------------------
# PROCESS CHUNKS
# ----------------------------
all_samples = []

for file_name in os.listdir(CHUNKS_FOLDER):
    if not file_name.endswith(".json"):
        continue

    path = os.path.join(CHUNKS_FOLDER, file_name)
    with open(path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

      # --- PICK RANDOM SUBSET ---
    subset_size = 5  # how many chunks you want to process from this file
    if len(chunks) > subset_size:
        chunks = random.sample(chunks, subset_size)
    # otherwise, keep all if fewer than subset_size

    for chunk in chunks:
        if isinstance(chunk, str):
            chunk = {"text": chunk}

        evidence_text = chunk["text"]

        # Prompt now includes the chunk text
        prompt = f"""
You are given a historical passage.

Generate EXACTLY ONE question that:
- Can be answered ONLY using the information in the passage
- Does NOT rely on external knowledge
- Does NOT refer to the passage, text, or document explicitly
- Is NOT a yes/no question

Then provide an answer grounded strictly in the passage.

Return STRICT JSON in the following format:

[
  {{
    "query": "...",
    "answer": "..."
  }}
]

Text:
\"\"\"
{evidence_text}
\"\"\"
"""


        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You are an expert historian who generates questions from historical passages."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.2,
                max_tokens=MAX_TOKENS
            )

            output_text = response.choices[0].message.content
            data = extract_json(output_text)

            if data:
                first_item = data[0]  # only take the first question-answer
                first_item["evidence_passages"] = [evidence_text]
                all_samples.append(first_item)


        except Exception as e:
            print(f"❌ Error for chunk: {e}")

# ----------------------------
# SAVE RESULTS
# ----------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    json.dump(all_samples, out, indent=4, ensure_ascii=False)

print(f"\n✅ RAG evaluation dataset saved to {OUTPUT_FILE}")
print(f"Total samples: {len(all_samples)}")



✅ RAG evaluation dataset saved to rag_eval.json
Total samples: 5


Embedding

In [7]:
import os
import json
# NEW: Partner package for HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings
# NEW: Community package for FAISS
from langchain_community.vectorstores import FAISS
# NEW: Standard Document class
from langchain_core.documents import Document


# --------- PATHS ----------
CHUNKS_FOLDER = "chunks_json"
FAISS_INDEX_PATH = "faiss_index"

# --------- EMBEDDINGS ----------
# langchain_huggingface is now the standard over langchain.embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded.")

# --------- LOAD CHUNK DATA ----------
documents = []

for filename in os.listdir(CHUNKS_FOLDER):
    if filename.endswith("_chunks.json"):
        file_path = os.path.join(CHUNKS_FOLDER, filename)
        print(f"📄 Loading: {file_path}")

        with open(file_path, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        for chunk in chunks:
            # NEW: Instead of a dict, we create a proper Document object
            doc = Document(
                page_content=chunk["text"],
                metadata={
                    "doc_id": chunk["doc_id"],
                    "chunk_id": chunk["chunk_id"],
                    "chunk_index": chunk["chunk_index"]
                }
            )
            documents.append(doc)

print(f"✅ Loaded {len(documents)} chunks")

# --------- CREATE & SAVE FAISS INDEX ----------
if documents:
    print("⏳ Creating FAISS index (this may take a minute)...")
    vector_store = FAISS.from_documents(documents, embedding_model)
    
    # Save the index locally so you don't have to re-embed next time
    vector_store.save_local(FAISS_INDEX_PATH)
    print(f"✅ FAISS index saved to '{FAISS_INDEX_PATH}'")

✅ Embedding model loaded.
📄 Loading: chunks_json\Topic 1_ Pre-16th Century Philippines Reading Materials_chunks.json
✅ Loaded 35 chunks
⏳ Creating FAISS index (this may take a minute)...
✅ FAISS index saved to 'faiss_index'


Retrieval

In [ ]:
import json
import os
# NEW: Import from the dedicated huggingface and community packages
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ---- Load Embeddings (MUST MATCH INDEX) ----
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded.")

# ---- Load FAISS Vector Store ----
FAISS_FOLDER = "faiss_index"

# Ensure the path exists before loading
if not os.path.exists(FAISS_FOLDER):
    raise FileNotFoundError(f"The folder {FAISS_FOLDER} does not exist.")

vectorstore = FAISS.load_local(
    FAISS_FOLDER,
    embeddings,
    allow_dangerous_deserialization=True  # Required for security in modern versions
)

# ---- Create Retriever ----
# k: number of final docs, fetch_k: number of docs to pass to MMR algorithm
retriever = vectorstore.as_retriever(
  
    search_kwargs={
        "k": 12,        # Final results returned
   # Docs to initially fetch for MMR to filter
    }
)

# ---- Load Evaluation Queries ----
with open("rag_eval.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# ---- Run Retrieval for Each Query ----
retrieval_results = []

for item in eval_data:
    query = item["query"]

    # MODERN CHANGE: Use .invoke() instead of .get_relevant_documents()
    docs = retriever.invoke(query)

    retrieval_results.append({
        "id": item.get("id"),
        "query": query,
        "retrieved_chunks": [
            {
                "content": doc.page_content,
                "metadata": doc.metadata
            }
            for doc in docs
        ]
    })

print(f"✅ Retrieved documents for {len(retrieval_results)} queries.")

# ---- Save Retrieval Output ----
OUTPUT_FILE = "retrieval_output.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, indent=2, ensure_ascii=False)

print(f"✅ Retrieval results saved to {OUTPUT_FILE}")

✅ Embedding model loaded.
✅ Retrieved documents for 5 queries.
✅ Retrieval results saved to retrieval_output.json


Retrieval Evaluation

In [21]:
import json

# ==========================================
# CONFIGURATION
# ==========================================
RETRIEVAL_FILE = "retrieval_output.json"   # The file you just generated
EVAL_FILE = "rag_eval.json" # Your gold standard dataset (check filename!)
OUTPUT_LOG = "evaluation_metrics.json"

# ==========================================
# 1. HELPER: TEXT NORMALIZATION
# ==========================================
def normalize_text(text):
    """
    Standardizes text by removing newlines, extra spaces, and lowercasing.
    Example: "Hello   World\n" -> "hello world"
    """
    if not text:
        return ""
    # Replace newlines/tabs with space, then collapse multiple spaces
    text = " ".join(text.split()) 
    return text.lower().strip()

# ==========================================
# 2. LOAD DATA
# ==========================================
print("📂 Loading data files...")

with open(RETRIEVAL_FILE, "r", encoding="utf-8") as f:
    retrieval_data = json.load(f)

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# Convert retrieval list to a dictionary for faster lookup (Key = Query)
# This handles cases where the order might differ
retrieval_map = {item['query']: item['retrieved_chunks'] for item in retrieval_data}

# ==========================================
# 3. RUN EVALUATION
# ==========================================
print("🚀 Running comparison...")

total_hits = 0
total_mrr = 0
total_queries = 0
detailed_logs = []

for eval_item in eval_data:
    query = eval_item['query']
    
    # "evidence_passages" or "evidence_text" depending on your exact json key
    # Based on your snippet, you used "evidence_passages" in one and "evidence_text" in another.
    # We try both to be safe.
    gold_passages = eval_item.get('evidence_passages', eval_item.get('evidence_text', []))
    
    # Skip if no evidence provided (rare)
    if not gold_passages:
        continue

    total_queries += 1

    # Get the retrieved docs for this specific query
    retrieved_docs = retrieval_map.get(query, [])
    
    is_hit = False
    rank = 0
    matched_passage = ""

    # Pre-normalize all retrieved chunks for this query
    norm_retrieved = [normalize_text(doc['content']) for doc in retrieved_docs]

    # CHECK: Does ANY gold passage exist inside the retrieved list?
    for gold in gold_passages:
        norm_gold = normalize_text(gold)
        
        # We check exact inclusion in the list
        # (Since you are using chunks, exact string match of the chunk text is best)
        if norm_gold in norm_retrieved:
            print(norm_gold)
            print(norm_retrieved)
            is_hit = True
            rank = norm_retrieved.index(norm_gold) + 1 # 1-based rank
            matched_passage = gold + "..." # Log snippet
            break
    
    # Update Metrics
    if is_hit:
        total_hits += 1
        total_mrr += (1.0 / rank)
        status = "HIT"
    else:
        status = "MISS"
        rank = 0

    # Log details
    detailed_logs.append({
        "query": query,
        "status": status,
        "rank": rank,
        "matched_passage": matched_passage
    })

# ==========================================
# 4. FINAL REPORT
# ==========================================
if total_queries > 0:
    hit_rate = total_hits / total_queries
    mrr_score = total_mrr / total_queries
else:
    hit_rate = 0
    mrr_score = 0

print("\n" + "="*40)
print("📊 RETRIEVAL METRICS REPORT")
print("="*40)
print(f"Total Queries Evaluated: {total_queries}")
print(f"Hit Rate (Recall):       {hit_rate:.2%}")
print(f"MRR Score:               {mrr_score:.4f}")
print("="*40)
print(f"✅ Detailed logs saved to {OUTPUT_LOG}")

with open(OUTPUT_LOG, "w", encoding="utf-8") as f:
    json.dump(detailed_logs, f, indent=4, ensure_ascii=False)

📂 Loading data files...
🚀 Running comparison...
occupying some coastal regions of the large islands at new guinea, the bismarcks and the solomons. finely decorated lapita pottery has been round in coastal or offshore island sites form the admiralties in the west to samoa in the east, a distance of about 5000 kilometers (see following chapters). this lapita expansion occurred between 16000 to 1000 bc and to north and east of the solomons it involved, for the first sustained period in the austronesian prehistory, the settlement of theses uninhabited regions continued onwards (irwin 1992), ultimately to incorporate all the islands of polynesia and micronesia and on the other side of the world, madagascar. 18 “the austronesians: historical and comparative perspective” this reading is a continuation of the previous excerpt from peter bellwood’s study on the austronesians. in this part, the focus is on the rationale behind the expansion and migration of the austronesians. for this, he cites 

Generation

In [15]:
from langchain_community.llms import Ollama

# ---- Initialize Ollama ----
llm = Ollama(model="mistral:instruct")

# ---- Generation Results Container ----
generation_results = []

for item in retrieval_results:
    query = item["query"]

    # ---- Combine Retrieved Context ----
    context = "\n\n".join(
        [chunk["content"] for chunk in item["retrieved_chunks"]]
    )

    # ---- Prompt Template ----
    prompt = f"""
You are an academic assistant.
Answer the question using ONLY the provided context.
If the answer is not present, say:
"The information is not available in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # ---- Generate Answer ----
    response = llm.invoke(prompt)

    generation_results.append({
        "id": item.get("id"),
        "query": query,
        "answer": response.strip(),
        "used_chunks": item["retrieved_chunks"]
    })

print(f"✅ Generated answers for {len(generation_results)} queries.")



with open("generation_output.json", "w", encoding="utf-8") as f:
    json.dump(generation_results, f, indent=2, ensure_ascii=False)



C:\Users\Migs\AppData\Local\Temp\ipykernel_3640\2654590259.py:4: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="mistral:instruct")


✅ Generated answers for 5 queries.


Automatic Metrics

In [17]:
import json
import numpy as np
import evaluate

# ==========================================
# 1. LOAD DATA
# ==========================================
# This should be your generation_output.json or the results from your previous script
with open("llm_judge_results.json", "r", encoding="utf-8") as f:
    results_data = json.load(f)

# Extract predictions and references
predictions = [item["generated_answer"] for item in results_data]
references = [item["reference_answer"] for item in results_data]

print(f"📊 Running metrics for {len(predictions)} samples...")

# ==========================================
# 2. INITIALIZE METRICS
# ==========================================
# 'evaluate' is the modern HuggingFace library for metrics
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# ==========================================
# 3. CALCULATE
# ==========================================

# BLEU Score
bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])

# ROUGE Score (L, 1, 2)
rouge_results = rouge.compute(predictions=predictions, references=references)

# BERTScore (Semantics-based)
# 'lang="en"' is required. It uses a RoBERTa model by default.
bert_results = bertscore.compute(
    predictions=predictions, 
    references=references, 
    lang="en",
    model_type="distilbert-base-uncased" # Faster/lighter than default
)

# ==========================================
# 4. AGGREGATE & SAVE
# ==========================================

# Calculate mean of BERTScore (it returns a list per sample)
avg_bert_f1 = np.mean(bert_results["f1"])

final_report = {
    "summary": {
        "sacrebleu": round(bleu_results["score"], 2),
        "rouge1": round(rouge_results["rouge1"], 4),
        "rouge2": round(rouge_results["rouge2"], 4),
        "rougeL": round(rouge_results["rougeL"], 4),
        "bertscore_f1_avg": round(float(avg_bert_f1), 4)
    },
    "per_sample_bertscore": bert_results["f1"]
}

print("\n" + "="*40)
print("📈 AUTOMATIC METRICS REPORT")
print("="*40)
print(f"SacreBLEU:  {final_report['summary']['sacrebleu']}")
print(f"ROUGE-L:    {final_report['summary']['rougeL']}")
print(f"BERTScore:  {final_report['summary']['bertscore_f1_avg']}")
print("="*40)

with open("auto_metrics_results.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=4)

print("✅ Metrics saved to auto_metrics_results.json")

📊 Running metrics for 5 samples...



📈 AUTOMATIC METRICS REPORT
SacreBLEU:  6.83
ROUGE-L:    0.1574
BERTScore:  0.7273
✅ Metrics saved to auto_metrics_results.json


Judge

In [16]:
import json
import re
from langchain_community.llms import Ollama

# ---- Load Files ----
with open("rag_eval_dataset.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

with open("generation_output.json", "r", encoding="utf-8") as f:
    gen_data = json.load(f)

# ---- Map eval answers by query ----
eval_map = {item["query"]: item for item in eval_data}

judge_llm = Ollama(model="mistral:instruct")


def judge_prompt(question, reference, generated, context):
    return f"""
You are an impartial academic evaluator.

Evaluate the GENERATED ANSWER based on:
- the QUESTION
- the REFERENCE ANSWER
- the PROVIDED CONTEXT

Question:
{question}

Reference Answer:
{reference}

Generated Answer:
{generated}

Provided Context:
{context}

Evaluation Criteria:
1. Correctness – Does the generated answer match the reference answer?
2. Faithfulness – Is the answer supported by the context?
3. Completeness – Does it fully answer the question without missing key points?

Scoring Rules:
- Scores range from 1 (poor) to 5 (excellent).
- If the generated answer contradicts the reference, correctness ≤ 2.
- If the answer adds unsupported claims, faithfulness ≤ 2.

Return ONLY valid JSON:

{{
  "correctness": <int>,
  "faithfulness": <int>,
  "completeness": <int>,
  "verdict": "<short explanation>"
}}
"""

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group()) if match else None

judge_results = []

for item in gen_data:
    query = item["query"]
    generated = item["answer"]

    eval_item = eval_map.get(query)
    if not eval_item:
        continue

    reference = eval_item["answer"]

    context = "\n\n".join(
        chunk["content"] for chunk in item["used_chunks"]
    )

    prompt = judge_prompt(query, reference, generated, context)

    verdict_text = judge_llm.invoke(prompt)
    scores = extract_json(verdict_text)

    judge_results.append({
        "query": query,
        "reference_answer": reference,
        "generated_answer": generated,
        "judge_scores": scores
    })

print(f"✅ Judged {len(judge_results)} answers.")

with open("llm_judge_results.json", "w", encoding="utf-8") as f:
    json.dump(judge_results, f, indent=2, ensure_ascii=False)


✅ Judged 5 answers.
